notebook นี้สำรวจชุดข้อมูล 38-Cloud ก่อนนำไปสร้าง pipeline การเทรน ครอบคลุมหัวข้อดังนี้:

1. โครงสร้างข้อมูลและการจัดเก็บไฟล์
2. ภาพตัวอย่าง (RGB, NIR, เฉลย)
3. การวิเคราะห์ความไม่สมดุลของคลาส (พิกเซลเมฆเทียบกับไม่ใช่เมฆ)
4. คุณภาพข้อมูล — การตรวจหาและกรองแพตช์ขยะ (nodata สูง)
5. แนวทางการปรับสเกลข้อมูล (normalization)
6. รหัสซีน (scene ID) และการวางแผนแบ่งชุด train/validation/test


## 1. การตั้งค่าเริ่มต้นและโครงสร้างข้อมูล (Setup & Data Structure)

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("CLOUD_PROJECT_ROOT", "."))
DATA_ROOT = PROJECT_ROOT / "38-Cloud_training"

print("Project root:", PROJECT_ROOT.resolve())
print("Data root:", DATA_ROOT.resolve())
print(os.listdir(DATA_ROOT))

# Prepare an outputs folder for saving figures generated in this notebook
outputs_dir = PROJECT_ROOT / "outputs"
outputs_dir.mkdir(exist_ok=True)


In [ ]:
import pandas as pd

all_patches_path = DATA_ROOT / "training_patches_38-Cloud.csv"
df_all_patches = pd.read_csv(all_patches_path)
patch_names_all = df_all_patches['name'].tolist()

print("จำนวนแพตช์ทั้งหมด:", len(patch_names_all))
print(df_all_patches.head())

In [ ]:
# เซลล์: เปิดไฟล์ .TIF ตัวอย่าง 1 ไฟล์ (แบนด์ red) ดูรายละเอียด
import rasterio

red_dir = DATA_ROOT / "train_red"
sample_files = os.listdir(red_dir)

print("จำนวนไฟล์ทั้งหมด:", len(sample_files))
print("ตัวอย่างชื่อไฟล์:", sample_files[0])

sample_path = red_dir / sample_files[0]
with rasterio.open(sample_path) as src:
    img = src.read(1)
    print("shape:", img.shape)
    print("dtype:", img.dtype)
    print("min:", img.min(), "max:", img.max())

In [ ]:
# เซลล์: เปิดไฟล์เฉลย (ground truth) ตัวอย่าง 1 ไฟล์

import numpy as np

gt_dir = DATA_ROOT / "train_gt"
gt_files = os.listdir(gt_dir)
print("ตัวอย่างชื่อไฟล์เฉลย:", gt_files[0])

gt_path = gt_dir / gt_files[0]
with rasterio.open(gt_path) as src:
    gt = src.read(1)
    print("shape:", gt.shape)
    print("dtype:", gt.dtype)
    print("unique values:", np.unique(gt))

In [ ]:
# เซลล์: เปรียบเทียบสถิติพิกเซล 4 แบนด์ ของแพตช์เดียวกัน

bands = ['red', 'green', 'blue', 'nir']
sample_name = "patch_100_5_by_12_LC08_L1TP_061017_20160720_20170223_01_T1.TIF"

for band in bands:
    band_dir = DATA_ROOT / f"train_{band}"
    path = band_dir / f"{band}_{sample_name}"
    with rasterio.open(path) as src:
        arr = src.read(1)
        print(f"{band}: min={arr.min()}, max={arr.max()}, mean={arr.mean():.1f}")

## 2. ภาพตัวอย่าง (RGB, NIR, เฉลย)

In [ ]:
# EDA: Sample images visualized in RGB and NIR

import matplotlib.pyplot as plt
import numpy as np

# เลือกตัวอย่าง 3 แพตช์แบบสุ่ม เพื่อแสดงความหลากหลาย
np.random.seed(42)
viz_samples = np.random.choice(patch_names_all, size=3, replace=False)

fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for row, patch_suffix in enumerate(viz_samples):
    # โหลด 4 แบนด์ของแพตช์นี้
    bands_data = {}
    for band in ['red', 'green', 'blue', 'nir']:
        path = DATA_ROOT / f"train_{band}" / f"{band}_{patch_suffix}.TIF"
        with rasterio.open(path) as src:
            bands_data[band] = src.read(1)
    
    # โหลดเฉลยคู่กัน
    gt_path = DATA_ROOT / "train_gt" / f"gt_{patch_suffix}.TIF"
    with rasterio.open(gt_path) as src:
        gt_mask = src.read(1)
    
    # ประกอบ RGB composite (normalize เพื่อแสดงผลเท่านั้น ไม่ใช่ normalize สำหรับเทรน)
    rgb = np.stack([bands_data['red'], bands_data['green'], bands_data['blue']], axis=-1).astype(np.float32)
    rgb_display = (rgb - rgb.min()) / (rgb.max() - rgb.min())
    
    # คอลัมน์ 1: RGB
    axes[row, 0].imshow(rgb_display)
    axes[row, 0].set_title(f"RGB — {patch_suffix[:30]}...")
    axes[row, 0].axis('off')
    
    # คอลัมน์ 2: NIR
    axes[row, 1].imshow(bands_data['nir'], cmap='gray')
    axes[row, 1].set_title("NIR band")
    axes[row, 1].axis('off')
    
    # คอลัมน์ 3: Ground truth mask
    axes[row, 2].imshow(gt_mask, cmap='gray')
    axes[row, 2].set_title("Ground truth cloud mask")
    axes[row, 2].axis('off')

plt.tight_layout()
plt.savefig(outputs_dir / 'sample_rgb_nir_gt.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. การวิเคราะห์ความไม่สมดุลของคลาส (Class Imbalance)

In [ ]:
# ==========================================
# เซลล์: เช็คสัดส่วนพิกเซลเมฆ vs ไม่ใช่เมฆ (สุ่มตัวอย่าง)
# ==========================================
import numpy as np

# สุ่มตัวอย่าง 30 แพตช์ (ไม่ต้องอ่านครบ 8400 ไฟล์ ใช้เวลานาน แค่สุ่มพอประเมินภาพรวมได้)
np.random.seed(42)
sample_gt_files = np.random.choice(gt_files, size=30, replace=False)

cloud_pixels = 0
total_pixels = 0

for fname in sample_gt_files:
    path = gt_dir / fname
    with rasterio.open(path) as src:
        arr = src.read(1)
        cloud_pixels += (arr == 255).sum()
        total_pixels += arr.size

cloud_ratio = cloud_pixels / total_pixels
print(f"สัดส่วนพิกเซลเมฆ: {cloud_ratio*100:.1f}%")
print(f"สัดส่วนพิกเซลไม่ใช่เมฆ: {(1-cloud_ratio)*100:.1f}%")

# เซลล์: ฮิสโตแกรมสัดส่วนเมฆต่อแพตช์

import matplotlib.pyplot as plt

# คำนวณสัดส่วนเมฆ "ต่อแพตช์" แทนที่จะรวมทุกแพตช์เป็นค่าเดียว
cloud_ratios_per_patch = []

for fname in sample_gt_files:
    path = gt_dir / fname
    with rasterio.open(path) as src:
        arr = src.read(1)
        ratio = (arr == 255).sum() / arr.size
        cloud_ratios_per_patch.append(ratio)

plt.figure(figsize=(8, 5))
plt.hist(cloud_ratios_per_patch, bins=15, color='steelblue', edgecolor='black')
plt.xlabel('Cloud pixel ratio per patch')
plt.ylabel('Number of patches')
plt.title('Cloud ratio distribution per patch (random sample, n=30)')
plt.savefig(outputs_dir / 'cloud_ratio_histogram.png')
plt.show()

# การวิเคราะห์ความไม่สมดุลของคลาส (Class Imbalance Analysis)

**การกระจายตัวระดับพิกเซล** (สุ่มตัวอย่าง 30 แพตช์จากชุดเทรน):
- พิกเซลเมฆ: 37.5%
- พิกเซลไม่ใช่เมฆ: 62.5%

ตัวเลขนี้เพียงอย่างเดียวดูเหมือนความไม่สมดุลจะไม่รุนแรงมาก แต่เมื่อดูการกระจายตัว
**ต่อแพตช์** กลับพบภาพที่ต่างออกไป — ฮิสโตแกรมของสัดส่วนเมฆต่อแพตช์แสดงรูปแบบ
**สองยอด (bimodal)** ชัดเจน คือแพตช์ส่วนใหญ่แทบไม่มีเมฆเลย (สัดส่วนใกล้ 0) หรือมีเมฆ
เกือบเต็มภาพ (สัดส่วนใกล้ 1) มีน้อยมากที่อยู่ตรงกลาง

**ผลต่อกลยุทธ์การเทรน:**
- Binary Cross-Entropy (BCE) เพียงอย่างเดียวอาจยังพอใช้ได้ในระดับหนึ่งเมื่อดูจากสัดส่วน
  ระดับพิกเซลที่ไม่รุนแรงนัก แต่การผสมกับ **Dice Loss** น่าจะให้ความทนทานมากกว่า เพราะ
  Dice loss วัดจากการซ้อนทับของพื้นที่โดยตรง และไวต่อการกระจายตัวแบบสองยอดระดับแพตช์
  น้อยกว่า
- ลักษณะสองยอดนี้ยังส่งผลต่อวิธีแบ่งชุด train/validation/test ด้วย — การแบ่งตามซีน
  (แทนที่จะสุ่มแบ่งรายแพตช์) ช่วยป้องกันไม่ให้ชุดใดชุดหนึ่งถูกครอบงำด้วยซีนที่มีแต่
  เมฆเยอะหรือไม่มีเมฆเลย

# 4. คุณภาพข้อมูล — การตรวจหาและกรองแพตช์ Nodata สูง

บางแพตช์มีพื้นที่ nodata จำนวนมาก ซึ่งเกิดจากภาพ Landsat-8 ที่เอียง
ถูกตัดเป็นแพตช์สี่เหลี่ยม เราวัดปัญหานี้ด้วย "สัดส่วนข้อมูลจริง" (informative ratio
คือสัดส่วนพิกเซลที่ไม่ใช่ศูนย์) แล้วกรองแพตช์ที่ต่ำกว่าเกณฑ์ออก

In [ ]:
# เซลล์: หาแพตช์ขยะที่ยังไม่ถูกกรอง (จากข้อมูลทั้งหมด 8,400 แพตช์)

import rasterio
import numpy as np

def informative_ratio(patch_name, data_root):
    """คำนวณสัดส่วนพิกเซลที่ 'มีข้อมูลจริง' (ไม่ใช่ 0)"""
    path = data_root / "train_red" / f"red_{patch_name}.TIF"
    with rasterio.open(path) as src:
        red = src.read(1)
    nonzero_ratio = (red != 0).sum() / red.size
    return nonzero_ratio

np.random.seed(42)
sample_patches = np.random.choice(patch_names_all, size=200, replace=False)

ratios = [informative_ratio(p, DATA_ROOT) for p in sample_patches]

low_info_count = sum(1 for r in ratios if r < 0.8)
print(f"จาก 200 แพตช์ตัวอย่าง (จากทั้งหมด 8,400): {low_info_count} แพตช์มีข้อมูลจริงน้อยกว่า 80%")
print(f"สัดส่วนข้อมูลจริงเฉลี่ย: {np.mean(ratios)*100:.1f}%")
print(f"ค่าต่ำสุด: {min(ratios)*100:.1f}%, ค่าสูงสุด: {max(ratios)*100:.1f}%")

In [ ]:
# เซลล์: สแกนครบทุกแพตช์ 

import time

start = time.time()
all_ratios = []

for i, patch_name in enumerate(patch_names_all):
    ratio = informative_ratio(patch_name, DATA_ROOT)
    all_ratios.append(ratio)
    if (i+1) % 1000 == 0:
        print(f"สแกนแล้ว {i+1}/{len(patch_names_all)} แพตช์...")

elapsed = time.time() - start
print(f"เสร็จใน {elapsed:.1f} วินาที")

THRESHOLD = 0.6  # ปรับได้ง่ายถ้าต้องการภายหลัง

result_df = pd.DataFrame({'name': patch_names_all, 'informative_ratio': all_ratios})
result_df['is_informative'] = result_df['informative_ratio'] >= THRESHOLD

print(f"\nแพตช์ที่ผ่านเกณฑ์ (>={THRESHOLD*100:.0f}% ข้อมูลจริง): {result_df['is_informative'].sum()}")
print(f"แพตช์ที่ไม่ผ่าน: {(~result_df['is_informative']).sum()}")

output_path = DATA_ROOT / "training_patches_custom_nonempty.csv"
result_df.to_csv(output_path, index=True)
print(f"บันทึกไฟล์แล้ว: {output_path.name}")

In [ ]:
# ค้นหาและเปิดไฟล์ทางการด้วย glob

import pandas as pd

official_files = list(DATA_ROOT.glob("*_nonempty.csv"))
print("File:", official_files)

nonempty_official_path = official_files[0]

df_nonempty_official = pd.read_csv(nonempty_official_path)
print("จำนวนแพตช์ในไฟล์นี้:", df_nonempty_official.shape)
print(df_nonempty_official.head())
print("\nชื่อคอลัมน์:", df_nonempty_official.columns.tolist())

## 5. แนวทางการปรับสเกลข้อมูล (Normalization Strategy)

In [ ]:
# EDA: Per-band pixel statistics (for normalization strategy)

# ==========================================
# โหลดรายชื่อแพตช์ที่ผ่านเกณฑ์กรอง (จากไฟล์ทางการ)
# ==========================================
official_files = list(DATA_ROOT.glob("*_nonempty.csv"))
print("ไฟล์ที่เจอ:", [f.name for f in official_files])

# ถ้ามีมากกว่า 1 ไฟล์ที่ตรง pattern ต้องเลือกให้ถูก
# (ถ้าลบไฟล์ custom เก่าที่ชื่อชนกันไปแล้ว ตรงนี้ควรเจอแค่ไฟล์เดียว)
nonempty_official_path = official_files[0]
df_nonempty_official = pd.read_csv(nonempty_official_path)
patch_names = df_nonempty_official['name'].tolist()

print("จำนวนแพตช์ที่ใช้ได้ (กรองแล้ว):", len(patch_names))

np.random.seed(42)
stats_sample = np.random.choice(patch_names, size=300, replace=False)  # ใช้ชุดที่กรองแล้ว

band_stats = {b: {'min': [], 'max': [], 'mean': []} for b in ['red', 'green', 'blue', 'nir']}

for patch_suffix in stats_sample:
    for band in ['red', 'green', 'blue', 'nir']:
        path = DATA_ROOT / f"train_{band}" / f"{band}_{patch_suffix}.TIF"
        with rasterio.open(path) as src:
            arr = src.read(1)
            band_stats[band]['min'].append(arr.min())
            band_stats[band]['max'].append(arr.max())
            band_stats[band]['mean'].append(arr.mean())

print(f"{'Band':<8}{'Min':<10}{'Max':<10}{'Mean':<10}")
for band in ['red', 'green', 'blue', 'nir']:
    b_min = min(band_stats[band]['min'])
    b_max = max(band_stats[band]['max'])
    b_mean = np.mean(band_stats[band]['mean'])
    print(f"{band:<8}{b_min:<10}{b_max:<10}{b_mean:<10.1f}")

In [ ]:
# ==========================================
# EDA: Compare percentile-based stats vs raw min/max
#      (excluding nodata pixels, which are 0)
# ==========================================
band_pixel_pools = {b: [] for b in ['red', 'green', 'blue', 'nir']}

for patch_suffix in stats_sample:
    for band in ['red', 'green', 'blue', 'nir']:
        path = DATA_ROOT / f"train_{band}" / f"{band}_{patch_suffix}.TIF"
        with rasterio.open(path) as src:
            arr = src.read(1)
            # เก็บเฉพาะพิกเซลที่ไม่ใช่ nodata (ไม่ใช่ 0)
            valid_pixels = arr[arr != 0]
            band_pixel_pools[band].append(valid_pixels)

print(f"{'Band':<8}{'RawMin':<10}{'RawMax':<10}{'P1':<10}{'P99':<10}")
for band in ['red', 'green', 'blue', 'nir']:
    all_pixels = np.concatenate(band_pixel_pools[band])
    raw_min = all_pixels.min()
    raw_max = all_pixels.max()
    p1 = np.percentile(all_pixels, 1)
    p99 = np.percentile(all_pixels, 99)
    print(f"{band:<8}{raw_min:<10}{raw_max:<10}{p1:<10.0f}{p99:<10.0f}")

## แนวทางการปรับสเกลข้อมูล (Normalization Strategy)

สถิติพิกเซลดิบ (ไม่รวม nodata, สุ่มตัวอย่าง 300 แพตช์) แสดงว่าการใช้ min/max จริง
สำหรับปรับสเกลมีปัญหา: แม้ตัดพิกเซล nodata ออกแล้ว ค่าสูงสุดยังคงสูงมากในทุกแบนด์
(57,000–65,535) ในขณะที่ช่วงเปอร์เซ็นไทล์ที่ 1–99 แคบกว่ามาก (เช่น Red: 6,168–40,018
เทียบกับค่าดิบ 5,030–61,694) แสดงถึงการมีพิกเซลค่าสุดขั้ว (outlier) อยู่จริง
(น่าจะมาจากหิมะ น้ำแข็ง หรือแสงสะท้อนจ้า) ซึ่งจะบีบอัดข้อมูลที่มีความหมายส่วนใหญ่
ให้อยู่ในช่วงแคบๆ ของสเกล 0–1 หากใช้ min/max ดิบตรงๆ

**การตัดสินใจ:** ปรับสเกลแต่ละแบนด์แยกกันโดยใช้ค่าเปอร์เซ็นไทล์ที่ 1 และ 99
(ตัดค่าให้อยู่ใน [0,1] หลังปรับสเกลแล้ว) แทนการใช้ min/max ดิบ ค่าเปอร์เซ็นไทล์นี้
จะคำนวณจากชุดเทรนเท่านั้น (เพื่อป้องกันข้อมูลรั่วไหลเข้าไปในชุด validation/test)
แล้วนำไปใช้เป็นค่าคงที่ตอน inference

**ขอบเขตการตัดค่ารายแบนด์ (จากตัวอย่าง 300 แพตช์ จะคำนวณใหม่จาก train split สุดท้ายอีกครั้ง):**

| แบนด์  | P1    | P99   |
|-------|-------|-------|
| Red   | 6168  | 40018 |
| Green | 6775  | 38524 |
| Blue  | 7754  | 39607 |
| NIR   | 6796  | 42143 |

## 6. รหัสซีน (Scene ID) และการวางแผนแบ่งชุด Train/Validation/Test

In [ ]:
# ==========================================
# EDA: Scene IDs (for train/val/test split planning)
# ==========================================
scene_csv_path = DATA_ROOT / "training_sceneids_38-Cloud.csv"
df_scenes = pd.read_csv(scene_csv_path)

print("จำนวน scene ทั้งหมด:", df_scenes.shape)
print(df_scenes.head())

scene_ids = df_scenes.iloc[:, 0].tolist()  # ดึงคอลัมน์แรกออกมาเป็น list
print("\nตัวอย่าง scene ID:", scene_ids[0])

In [ ]:
# ==========================================
# EDA: จำนวนแพตช์ (กรองแล้ว) ต่อ scene
# ==========================================
def find_scene_in_patch(patch_name, scene_list):
    """หาว่าแพตช์นี้มาจาก scene ไหน"""
    for scene in scene_list:
        if scene in patch_name:
            return scene
    return None

# เช็คเฉพาะ patch_names (ชุดที่กรองแล้ว 5,155 แพตช์)
patch_to_scene = {p: find_scene_in_patch(p, scene_ids) for p in patch_names}

scene_patch_counts = pd.Series(patch_to_scene.values()).value_counts()
print("จำนวนแพตช์ (กรองแล้ว) ต่อ scene:")
print(scene_patch_counts.sort_index())
print("\nน้อยสุด:", scene_patch_counts.min(), " มากสุด:", scene_patch_counts.max())

## 7. สรุปข้อค้นพบสำคัญและการตัดสินใจ

- **คุณภาพข้อมูล:** ประมาณ 46% ของแพตช์เทรนทั้งหมด 8,400 แพตช์ มีพื้นที่ nodata
  จำนวนมาก ใช้ไฟล์กรองทางการ (`training_patches_38-cloud_nonempty.csv`, 5,155 แพตช์)
  เป็นแหล่งหลัก ส่วนไฟล์กรองที่คำนวณเอง (เกณฑ์=0.6, 4,694 แพตช์) เก็บไว้เป็นทางเลือก
  สำรอง เลือกได้ผ่านพารามิเตอร์ `filter_source` ใน `dataset.py`
- **ความไม่สมดุลของคลาส:** สัดส่วนเมฆระดับพิกเซล (~37.5%) ดูเหมือนไม่สมดุลรุนแรงมาก
  แต่การกระจายตัว**ต่อแพตช์**เป็นแบบ**สองยอด (bimodal)** ชัดเจน — แพตช์ส่วนใหญ่แทบ
  ไม่มีเมฆเลยหรือมีเมฆเกือบเต็มภาพ สิ่งนี้เป็นแรงจูงใจให้ผสม BCE กับ Dice Loss
  ตอนเทรน
- **การปรับสเกลข้อมูล:** min/max ดิบไม่เหมาะสมเพราะมีค่าสุดขั้ว (สูงถึง 65,535
  ซึ่งเป็นเพดานของ uint16) แม้ตัดพิกเซล nodata ออกแล้ว แต่ละแบนด์ถูกปรับสเกลแยกกัน
  ด้วยเปอร์เซ็นไทล์ที่ 1/99 คำนวณ**จากชุดเทรนเท่านั้น**เพื่อป้องกันข้อมูลรั่วไหล
  แล้วตัดค่าให้อยู่ในช่วง [0, 1]
- **การแบ่งชุด train/val/test:** แบ่งแพตช์ตาม**ซีน** (12/3/3 จากทั้งหมด 18 ซีน)
  ไม่ใช่การสุ่มแบ่งรายแพตช์ เพราะแพตช์จากซีนเดียวกันมีความคล้ายกันสูงมาก จำนวนแพตช์
  ต่อซีนใกล้เคียงกันมาก (284–288) ทำให้การแบ่งแบบนี้ได้สัดส่วนที่สมดุลโดยธรรมชาติ

การตัดสินใจเหล่านี้ถูก implement ไว้ใน `dataset.py` และอ้างอิงถึงใน `REPORT.md`